# 3D Fluorescence LIVE/DEAD Segmentation (v1.2.2)

Segments LIVE/DEAD-stained 3D (or 2D) images and calls every cell LIVE or DEAD.

> Python 3.10–3.14. Install: `uv pip install -r requirements.txt --override overrides.txt`. On 3.14, StarDist and `.czi` files are unavailable.

**How to use**
1. Set the input in **Cell 3**: one stacked image (`input_file`) **or** a separate LIVE and DEAD image (`input_file_live` + `input_file_dead`). Also set the ROI, experiment name and sizes. Set exports in **Cell 4**.
2. Run Cell 5 and fill in the stain table: LIVE and DEAD, plus any extra stain (set each marker as nuclear or cytoplasmic). A counterstain that should make a cell LIVE (e.g. Hoechst) goes in with **Add LIVE stain** (`LIVE_2`). Click **Confirm**.
3. Run the rest. The processing steps of Cells 9-14 are optional (see Section 4).

**Input.** A stacked image holds every channel in one file. Separate LIVE and DEAD images (same size and voxel size) are read as one image, with the LIVE file's channels first (`'LIVE - ...'` in the stain table), then the DEAD file's (`'DEAD - ...'`). Folders run **batch mode**: one folder of stacked images, or a LIVE folder and a DEAD folder, paired in name order.

**The call.** A cell positive for a DEAD marker is DEAD, one positive only for a LIVE marker (LIVE, `LIVE_2`, …) is LIVE (`ld_rule = "dead_priority"`). So with Hoechst as `LIVE_2`: Hoechst only → LIVE, Hoechst + DEAD → DEAD. A cell positive for no LIVE and no DEAD marker is **UNCLASSIFIED**: reported, but left out of the viability, LIVE / (LIVE + DEAD).

**Bright signal only.** LIVE/DEAD stains are binary: a cell is clearly bright or it is not. After the threshold, the masks of the call markers keep only objects that stand out clearly from the background, and their dim fringe becomes background (Cell 15; `ld_min_brightness`, `ld_edge_brightness`, `ld_min_snr` in Cell 3). This drops out-of-focus cells, in 2D images often light from another depth.

**Segmentation settings.** The validated settings of NucleoSeg3D, by default the version that scores best on the manual ground truths (`version="best"` in Cell 4; now v1.6.41, `settings/optimal_settings_v1.6.41.json`: nuclei mean F1 0.56, F1@0.5 0.77 on six annotated images). `version="1.6.4"` gives the settings this notebook used before (mean F1 0.51, F1@0.5 0.74 on four images). The nuclei are segmented from the **nuclear** markers (e.g. DEAD = EthD-1 / PI, a Hoechst counterstain), so the per-image steps act on them:
- the **contrast of every nuclear marker** comes from its own histogram (Cell 8);
- v1.6.41 settings: the **nuclei edge fraction** follows the signal-to-noise ratio of the nuclear markers' histogram (Cell 8);
- the **nucleus size**: `nuclei_diameter` in Cell 3 is only a starting value. The typical nucleus is measured in the nuclear markers and the sizes adapted to it (Cell 8; 2D images: used as given, only the resolution is adapted);
- the nuclei get the validated steps (haze removal, watershed, size refinement, Z split, edges). The rescue of missed nuclei is off: positivity is read from the threshold masks, so a rescued nucleus would have no call;
- the cell bodies (**cytoplasmic** markers, e.g. LIVE = calcein) get the validated smoothing, threshold, watershed split and size refinement.

Every other setting is fixed. To change one anyway, name it in `custom_settings` (Cell 4) and re-run from Cell 4. The LIVE/DEAD call (`min_positive_fraction`, `ld_rule`) and the bright-signal filter (Cell 3) are experiment choices.

**Methods:** watershed (default, the validated one), Cellpose (`trig_cellpose`, `trig_cellpose_cyto`), StarDist for nuclei (`trig_stardist`).

# 1. Import Libraries and Helpers

**Cell 1** — Import libraries (prints the Python version).

In [ ]:
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

**Cell 2** — Load helper functions. Re-run after editing `helpers/notebook_helpers.py`.

In [ ]:
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

The only cells to edit for a new dataset.

**Cell 3** — Settings. Give **either** one stacked image (`input_file`) **or** a separate LIVE and DEAD image (`input_file_live` + `input_file_dead`, then `input_file` is ignored). A folder runs **batch mode**: a folder of stacked images, or a LIVE folder and a DEAD folder, paired in name order (the pairs are printed, check them). Each image gets its own nuclear-marker contrast and nucleus size. Each setting is explained in line.

In [ ]:
# --- Input: one stacked image (all channels in one file) ... ---
input_file = 'path/to/your_image_file.nd2'  # file (.nd2/.tif/.czi/.lif/.oib) or folder (batch)
# --- ... or separate LIVE and DEAD images (None = use input_file) ---
input_file_live = None  # LIVE image file, or folder of LIVE images (batch)
input_file_dead = None  # DEAD image file, or folder of DEAD images (batch)
input_file = combine_live_dead_inputs(input_file, input_file_live, input_file_dead)

ROI = [0, 0, 0, 0, 0, 0]  # [x0, x1, y0, y1, z0, z1]; 0 = full range
interactive_roi = False   # True = draw ROI in napari

# --- Experiment ---
name_setup = 'your_experiment_name'  # prefix of saved files
use_setup = True                     # reuse saved stain table and contrast
calibrated_contrast = True           # non-nuclear markers: True = contrast from each channel's histogram (calibrated, no napari window), False = set it yourself in napari (Cell 8)
                                     # (nuclear markers: always from their histogram, Cell 8)

# --- Sizes ---
nuclei_diameter = 8   # um, approximate: measured in the nuclear markers (Cell 8); 2D images: used as given
cell_diameter = 20    # um, for the cytoplasmic markers

# --- Segmentation method (all False = watershed, the validated method) ---
trig_cellpose = False       # Cellpose for nuclei (GPU recommended)
trig_stardist = False       # StarDist for nuclei (not on Python 3.14)
trig_cellpose_cyto = False  # Cellpose for cell bodies

# --- Quantification ---
multilabel = True  # multi-marker combinations in summary (e.g. LIVE + DEAD)

# --- LIVE/DEAD signal: only clearly bright cells count (Cell 15) ---
ld_min_brightness = 0.3    # LIVE/DEAD objects dimmer than this x the typical bright cell (above background) are discarded (0 = off)
ld_edge_brightness = 0.15  # LIVE/DEAD voxels dimmer than this x the typical bright cell are background (0 = off)
ld_min_snr = 5.0           # an object must also stand >= this x the noise above background (0 = off)

# --- LIVE/DEAD call ---
min_positive_fraction = 0.2  # positive = marker covers >= this fraction of the compartment
ld_rule = "dead_priority"    # "dead_priority" (DEAD wins) or "coverage" (higher coverage wins)

**Cell 4** — Exports and segmentation settings, used in single-file and batch mode. The segmentation settings are the validated ones of NucleoSeg3D: `version="best"` (default) takes the settings file that scores best on the manual ground truths (now v1.6.41), `version="1.6.4"` the settings this notebook used before. One change for LIVE/DEAD: the rescue of nuclei the threshold missed is off. Positivity is read from the threshold masks, so a rescued nucleus would be positive for no marker. `custom_settings` changes one by name (e.g. `{"nuclei_edge_fraction": 0.3}`; names in `settings/optimal_settings_v<version>.json`).

**Resolution.** By default the image is resampled so that a nucleus spans 12–19 voxels (Cell 8). To choose it yourself, set `"scale_factor"` in `custom_settings` (e.g. `0.75`): it is then used as given, also in batch mode. The run time grows with the number of voxels, about as `scale_factor`³ in 3D.

In [ ]:
# --- 3D exports (skipped for 2D images) ---
export_vtk = True      # VTK volumes for ParaView (slow)
export_stl = True      # STL mesh per marker (slow)
export_fea = True      # FEA mesh of the whole cells, Abaqus .inp (slow)

# --- Segmentation settings: validated, normally left as they are ---
custom_settings = {}   # e.g. {"nuclei_edge_fraction": 0.3}; {} = all validated values
# Resolution: by default set per image so a nucleus spans 12-19 voxels (Cell 8).
# A scale_factor given here is used as given, e.g. {"scale_factor": 0.75}
# (finer = more detail, slower; 1.0 = full resolution).

from helpers.optimal_settings_ld import (
    load_ld_settings, set_nuclear_contrast, adapt_to_image, correct_depth, remove_haze,
    nuclei_settings, cell_body_settings, ld_batch_settings, settings_for_report,
    adapt_to_histograms)
# version: "best" = the best-scoring validated settings (now v1.6.41), or "1.6.4"
opt = load_ld_settings(custom_settings, version="best")  # rescue of missed nuclei off

**Cell 5** — Load image and stain table. Single-plane images are handled as 2D (`is_2d`).

The table starts with the two LIVE/DEAD rows: pick each one's channel, color, marker and **localization** (nuclear / cytoplasmic). Extra stains take the next unused channel:
- **Add LIVE stain** (`LIVE_2`, …): another LIVE marker, e.g. a **Hoechst** counterstain. A cell positive only for it is LIVE; with a DEAD marker too it is DEAD.
- **Add DEAD stain** (`DEAD_2`, …): another DEAD marker.
- **Add other stain**: segmented and quantified, but not in the LIVE/DEAD call.

Channels left out are listed below the table and dropped from the analysis. With separate LIVE and DEAD images, the channels are named `'LIVE - ...'` and `'DEAD - ...'` after their file. Click **Confirm**.

In [ ]:
batch_mode = is_batch_input(input_file)
file_meta = None
is_2d = False  # set from the loaded image
stain_table_path = f"{name_setup}_ld_stain_dict.json" if use_setup else None

if batch_mode:
    print(f"Batch mode: {len(list_image_files(input_file))} image(s) will be "
          f"processed by Cell 6.")
else:
    if interactive_roi:
        ROI = select_roi_interactively(input_file, ROI, napari_module=napari,
                                       stain_table_path=stain_table_path)
    (meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
     _, _, zooms, used_lazy_loading) = initialize_dataset(  # cyto/PCM factors unused
        input_file, ROI,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=opt["scale_factor"],
        zoom_factors=opt["zoom_factors"],
    )
    is_2d = img.shape[0] == 1
    print("2D image: segmentation in 2D, sizes as areas [um2], 3D exports skipped."
          if is_2d else f"3D image: {img.shape[0]} Z planes.")

channel_list = print_channel_list(input_file, file_meta=file_meta)
stain_editor = edit_ld_stain_dict(channel_list, name_setup=name_setup, use_saved=use_setup)

**Cell 6** — Read the stain table. In batch mode, processes the whole folder (to `<folder>_output/`, plus `live_dead_batch_summary.xlsx`) with the settings of Cells 3–4 and stops.

In [ ]:
stain_dict, localization = stain_editor.result()
nuclear_conditions = [c for c, loc in localization.items() if loc == 'nuclear']
print(f"stain_dict = {stain_dict}")
print(f"localization = {localization}")
print(f"nuclear markers (nuclei segmentation) = {nuclear_conditions}")

if batch_mode:
    batch_summary = run_ld_batch_folder(
        input_file,
        roi_coords=ROI,
        stain_dict=stain_dict,
        localization=localization,
        name_setup=name_setup,
        use_setup=use_setup,
        calibrated_contrast=calibrated_contrast,
        interactive_roi=interactive_roi,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        trig_cellpose=trig_cellpose,
        trig_stardist=trig_stardist,
        trig_cellpose_cyto=trig_cellpose_cyto,
        multilabel=multilabel,
        min_positive_fraction=min_positive_fraction,
        ld_rule=ld_rule,
        ld_min_brightness=ld_min_brightness,
        ld_edge_brightness=ld_edge_brightness,
        ld_min_snr=ld_min_snr,
        export_vtk=export_vtk,
        export_stl=export_stl,
        export_fea=export_fea,
        **ld_batch_settings(opt),  # validated segmentation settings (Cell 4)
        settings=settings,
        napari_module=napari,
        progress=tqdm,
    )
    display(batch_summary)
    raise BatchCompleted("Batch processing finished -- the remaining cells are for single-file mode only.")
else:
    print("Single-file mode -- continue with the next cells.")

# 3. Preparation and Preview

**Cell 7** — Prepare the image stack and preview the channels in napari.

In [ ]:
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, r_xyz=(r_X, r_Y, r_Z), progress=tqdm,
)

**Cell 8** — Contrast and gamma per channel: reused from `<name_setup>_setup.csv` if it exists, otherwise calibrated from each channel's histogram (`calibrated_contrast = True`, the default, no napari window) or set yourself in napari (`calibrated_contrast = False`). The contrast of every **nuclear** marker is then set from its own histogram (the rule the segmentation was validated with; gamma 1.15 with the v1.6.41 settings). With the v1.6.41 settings, the nuclei edge fraction is read off the histogram of the merged nuclear markers (printed with its value). The typical nucleus is measured in the nuclear markers, and `nuclei_diameter`, the resolution and the µm sizes are adapted to it. The stain settings table is shown once the napari window is closed.

In [ ]:
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    calibrated_contrast=calibrated_contrast,
    suggest_contrast=opt["suggest_contrast"],
    settings=settings,
    napari_module=napari,
    r_xyz=(r_X, r_Y, r_Z),
    progress=tqdm,
)
set_nuclear_contrast(im_final_stack['Original image'], stain_complete_df,
                     original_stain_complete_df, opt, nuclear_conditions)
adapt_to_histograms(im_final_stack['Original image'], stain_complete_df, opt,
                    nuclear_conditions)  # v1.6.41 settings: nuclei edge fraction
display(stain_complete_df)

nuclei_diameter, zooms, adapt_changes = adapt_to_image(
    im_final_stack['Original image'], stain_complete_df, (r_Z, r_Y, r_X),
    nuclei_diameter, opt, nuclear_conditions, is_2d=is_2d)
im_final_stack = reset_processing_steps(im_final_stack)  # Cells 9-14 start from the original image

# 4. Image Processing and Segmentation

Run in order. The settings are the validated ones of Cell 4.

**Optional processing steps.** Cells 9–14 can each be skipped: a skipped cell passes its input on unchanged, and the next cell run works on the last image computed. E.g. skip Cell 11 to go from the resampled image (Cell 10) to the contrast (Cell 12), or skip Cells 9–14 to threshold the original image of Cell 8 directly. Cell 15 lists the skipped steps (they are also saved in the report). Without resampling (Cell 10) the image keeps its original voxel size. Skipping any step departs from the validated settings. To change which steps run, re-run from Cell 8.

**Cell 9** *(optional)* — Normalize each channel to 0–255 (after the depth correction, if switched on in `custom_settings`; off in the validated settings).

In [ ]:
im_final_stack = correct_depth(im_final_stack, stain_complete_df, opt, nuclear_conditions, voxel_z=r_Z)
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 10** *(optional)* — Resample to isotropic voxels. Skipped: the image keeps its original voxel size.

In [ ]:
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zooms,
    meta=meta,
    edge_mode=opt["resample_edge_mode"],  # v1.6.41 settings: keeps the last Z slice
)

**Cell 11** *(optional)* — Denoise (median filter), then remove the out-of-focus haze of the nuclear markers slice by slice.

In [ ]:
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)
im_final_stack = remove_haze(im_final_stack, stain_complete_df, opt, (r_zZ, r_zY, r_zX),
                             nuclear_conditions)

**Cell 12** *(optional)* — Apply contrast and gamma from Cell 8.

In [ ]:
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 13** *(optional)* — Gaussian smoothing.

In [ ]:
im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=opt["sigma"],
)

**Cell 14** *(optional)* — Contrast enhancement (histogram equalization).

In [ ]:
im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=opt["num_plateaus"],
    plateau_factor=opt["plateau_factor"],
)

**Cell 15** — Threshold every channel (global + local), on the last image computed by Cells 9–14 (the skipped steps are listed first). Nuclear masks → nuclei, cytoplasmic masks → cell bodies, all → marker positivity.

LIVE/DEAD stains are binary: a cell is clearly bright or it is not. The masks of the call markers (LIVE, DEAD and any `LIVE_2` / `DEAD_2`, e.g. Hoechst) then keep only objects that stand out clearly, using `ld_min_brightness` × the typical bright cell and `ld_min_snr` × the noise above background. Their dim fringe becomes background (`ld_edge_brightness`). This drops out-of-focus cells (in 2D images, often light from another depth) and halos. The table printed below shows what was kept and removed per channel. Other stains keep the plain threshold.

In [ ]:
im_final_stack, zooms, r_zX, r_zY, r_zZ, skipped_steps = complete_skipped_steps(
    im_final_stack, zoom_factors=zooms, r_xyz=(r_X, r_Y, r_Z))  # skipped Cells 9-14
im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=opt["threshold_method"],
    global_threshold_weight=opt["global_threshold_weight"],
    sauvola_weight=opt["sauvola_weight"],
    nuclear_conditions=nuclear_conditions,
    progress=tqdm,
)
im_final_stack = ld_binary_threshold(  # clearly bright LIVE/DEAD signal only
    im_final_stack, stain_complete_df, nuclei_diameter, (r_zX, r_zY, r_zZ),
    nuclear_conditions=nuclear_conditions,
    min_brightness=ld_min_brightness,
    edge_brightness=ld_edge_brightness,
    min_snr=ld_min_snr,
)

**Cell 16** — Export histograms and parameters to `<file>_histograms.xlsx`.

In [ ]:
ld_settings = {
    'Input LIVE file': input_file_live or '(stacked image)',
    'Input DEAD file': input_file_dead or '(stacked image)',
    'Localization': localization,
    'Nuclei diameter (um)': nuclei_diameter,
    'Cell diameter (um)': cell_diameter,
    'Use Cellpose (nuclei)': trig_cellpose,
    'Use StarDist (nuclei)': trig_stardist,
    'Use Cellpose (cell bodies)': trig_cellpose_cyto,
    'LD min brightness': ld_min_brightness,
    'LD edge brightness': ld_edge_brightness,
    'LD min SNR': ld_min_snr,
    'Min positive fraction': min_positive_fraction,
    'LIVE/DEAD rule': ld_rule,
    'Skipped processing steps': '; '.join(skipped_steps) or 'none',
    **settings_for_report(opt, adapt_changes),
}

output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    zoom_factors=zooms, scale_factor=opt["scale_factor"],
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    trig_cellpose_cyto=trig_cellpose_cyto,
    multilabel=multilabel,
    sigma=opt["sigma"], num_plateaus=opt["num_plateaus"],
    plateau_factor=opt["plateau_factor"], threshold_method=opt["threshold_method"],
    global_threshold_weight=opt["global_threshold_weight"],
    sauvola_weight=opt["sauvola_weight"],
    nuclei_split_config=opt["nuclei_split_config"],
    aggregate_distance_factor=None,  # no aggregates / PCM in the LIVE/DEAD analysis
    extra_params=ld_settings,
    progress=tqdm,
)

**Cell 17** — Segment nuclei from the nuclear markers with the chosen method and the validated settings: watershed, size refinement (merge fragments, re-split oversized nuclei), split of nuclei stacked in Z, each nucleus edge redrawn relative to its own brightness. Skipped if there are no nuclear markers.

In [ ]:
im_segmentation_stack = segment_ld_nuclei(
    im_final_stack,
    stain_df=stain_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    **nuclei_settings(opt),  # validated settings (Cell 4)
    progress=tqdm,
)

**Cell 18** — Segment cell bodies from the cytoplasmic markers (validated watershed split and size refinement, at the scale of `cell_diameter`), then combine them with the nuclei into one label per cell. Cells without a stained nucleus get an estimated one.

In [ ]:
im_segmentation_stack, stain_complete_df = segment_ld_cells(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    trig_cellpose_cyto=trig_cellpose_cyto,
    **cell_body_settings(opt),  # validated settings (Cell 4)
    progress=tqdm,
)

**Cell 19** — LIVE/DEAD call: each marker is measured in its compartment, and each cell is called with `ld_rule`. A cell is LIVE-positive if it is positive for any LIVE marker (LIVE, `LIVE_2`, …) and DEAD-positive for any DEAD marker. With `dead_priority`: DEAD-positive → DEAD (e.g. Hoechst + DEAD), LIVE-positive only → LIVE (e.g. Hoechst only). Cells positive for no LIVE and no DEAD marker are **UNCLASSIFIED**: reported, but left out of the viability, LIVE / (LIVE + DEAD). Result: `ld_cells_df`.

In [ ]:
im_segmentation_stack, ld_cells_df = classify_live_dead(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    min_positive_fraction=min_positive_fraction,
    ld_rule=ld_rule,
    progress=tqdm,
)
ld_cells_df

**Cell 20** — View processing, segmentation and LIVE (green) / DEAD (red) cells in napari. The processing viewer shows the original image and only the steps of Cells 9–14 that were run.

In [ ]:
viewer_0, viewer_1 = view_ld_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

Intensities are reported for **positive cells** and **all cells**.

**Cell 21** — Measure intensity, size and position of every cell (`labels_df`).

In [ ]:
percell_mean_df, percell_std_df = compute_percell_marker_intensity_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    progress=tqdm,
)

labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    multilabel=multilabel,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 22** — LIVE/DEAD summary: counts, viability, marker positivity and sizes.

In [ ]:
print_ld_summary(
    ld_cells_df,
    stain_df=stain_df,
)

**Cell 23** — Full table for export.

In [ ]:
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 24** — Per-cell intensity report (PDF).

In [ ]:
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack, im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    show_plot=False,  # KDE plots go to the PDF only
    progress=tqdm,
)

**Cell 25** — Spatial distribution of LIVE/DEAD cells and viability along X, Y, Z.

In [ ]:
plot_ld_spatial_distributions(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
    volume_shape=im_final_stack['Filtered image'].shape,
    r_zxyz=(r_zX, r_zY, r_zZ),
)

**Cell 26** — Cell and nucleus size of LIVE and DEAD cells.

In [ ]:
plot_ld_size_distributions(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
)

**Cell 27** — LIVE/DEAD counts and LIVE vs DEAD coverage per cell. Use it to check `min_positive_fraction`.

In [ ]:
plot_live_dead_classification(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
)

**Cell 28** — Marker intensity vs cell size (flow-cytometry style).

In [ ]:
plot_marker_intensity_clouds(
    percell_mean_df,
    stain_complete_df=stain_complete_df,
    im_segmentation_stack=im_segmentation_stack,
    r_xyz=(r_zX, r_zY, r_zZ),
    stain_df=stain_df,
    progress=tqdm,
    input_file=input_file,
    is_2d=is_2d,
)

# 6. Export

3D exports are skipped for 2D images. Which ones run is set in Cell 4; the single-cell export is set in Cell 31.

**Cell 29** — VTK volumes for ParaView (`export_vtk`).

In [ ]:
if not export_vtk:
    print("VTK volumes skipped (export_vtk = False).")
elif is_2d:
    print("2D image: VTK volumes skipped (3D only).")
else:
    build_vtk_volumes(
        im_segmentation_stack,
        labels_full_df=labels_full_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 30** — STL mesh per marker (`export_stl`).

In [ ]:
if not export_stl:
    print("STL meshes skipped (export_stl = False).")
elif is_2d:
    print("2D image: STL meshes skipped (3D only).")
else:
    export_marker_stl(
        im_segmentation_stack,
        stain_df=stain_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 31** — Single-cell VTK export. Set `export_nuc_3D_vtk = True` in this cell and run it. It asks for the cell label (`Cell label` in `ld_cells_df`).

In [ ]:
export_nuc_3D_vtk = False  # True = export one cell as VTK, asks for its label

if not export_nuc_3D_vtk:
    print("Single-cell VTK export skipped (export_nuc_3D_vtk = False).")
elif is_2d:
    print("2D image: single-cell VTK export skipped (3D only).")
else:
    nuc_label = int(input("Enter cell label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        zoom_factors=zooms,
    )

**Cell 32** — Excel report (`<file>_live_dead.xlsx`).

In [ ]:
output_path = export_live_dead_to_excel(
    input_file,
    original_stain_complete_df,
    ld_cells_df,
    parameters=ld_settings,
)

**Cell 33** — FEA mesh of the whole cells (Abaqus `.inp`; `export_fea`).

In [ ]:
if not export_fea:
    print("FEA mesh skipped (export_fea = False).")
elif is_2d:
    print("2D image: FEA mesh skipped (3D only).")
else:
    export_fea_mesh(
        {'Nuclei': im_segmentation_stack['Cytoplasm']},  # whole cells (many nuclei are estimated)
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        progress=tqdm,
    )